# 02 -- Layer 1 Parameter Justification

Every Layer 1 parameter (k, eps, minPts, Mahalanobis threshold) derived label-free --
none of this touches the Answer Set. Full rationale and rejected alternatives in
`docs/methodology.md`. Cells here are diagnostic; they validate `awik.config` defaults,
they don't change them.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                              normalized_mutual_info_score)
from sklearn.neighbors import NearestNeighbors

from awik.config import DEFAULT_K, DEFAULT_EPS, DEFAULT_MINPTS, DEFAULT_MAH_THRESH
from awik.features import ALL_FEATURES
from awik.stats_utils import knee_of_curve, sampled_weeks
from awik import io

df_master = io.load_df('01_df_master')
weeks_df = io.load_df('01_weeks_df')

## k optimal (elbow + silhouette, single sample week)

In [ ]:
yr_s, wk_s = 2010, 20
df_s = df_master[(df_master['Year']==yr_s) & (df_master['Week']==wk_s)].copy()
Xs  = StandardScaler().fit_transform(df_s[ALL_FEATURES])
Xps = PCA(n_components=2, random_state=42).fit_transform(Xs)

k_range = range(2, 9)
wcss, sil = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lb = km.fit_predict(Xps)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(Xps, lb) if k > 1 else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("k Optimal -- Sample 2010-W20", fontweight="bold")
axes[0].plot(list(k_range), wcss, "bo-", linewidth=2, markersize=8)
axes[0].axvline(4, color="red", ls="--", label="k=4 (elbow)")
axes[0].set_title("Elbow Method (WCSS)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("WCSS"); axes[0].legend()
axes[1].plot(list(k_range), sil, "go-", linewidth=2, markersize=8)
axes[1].axvline(4, color="red", ls="--", label="k=4")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette (higher = better)"); axes[1].legend()
plt.tight_layout(); plt.show()

OPTIMAL_K = DEFAULT_K
print(f"=> k optimal: {OPTIMAL_K} | Silhouette: {sil[OPTIMAL_K-2]:.4f}")

In [ ]:
# PCA explained variance -- Section 3.3 justification
pca_full = PCA(random_state=42).fit(Xs)
ev = pca_full.explained_variance_ratio_
print(f"PC1: {ev[0]:.1%} | PC2: {ev[1]:.1%} | PC1+PC2: {ev[0]+ev[1]:.1%}")
print(f"Components for 90% variance: {(ev.cumsum() < 0.90).sum() + 1}")

evs = []
for _, row in sampled_weeks(weeks_df).iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50: continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    pca_w = PCA(random_state=42).fit(Xw)
    evs.append(pca_w.explained_variance_ratio_[:2].sum())

print(f"Median PC1+PC2 (16 weeks): {np.median(evs):.1%}")
print(f"IQR: [{np.percentile(evs,25):.1%}, {np.percentile(evs,75):.1%}]")

## PCA structure figure (Section 3.3 / Section 4)

In [ ]:
sample_idx = list(range(0, len(weeks_df), 5))
all_ev = []
for _, row in weeks_df.iloc[sample_idx].iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50: continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    pca_w = PCA(random_state=42).fit(Xw)
    all_ev.append(pca_w.explained_variance_ratio_)
all_ev = np.array(all_ev)
ev_median = np.median(all_ev, axis=0)
ev_q25 = np.percentile(all_ev, 25, axis=0)
ev_q75 = np.percentile(all_ev, 75, axis=0)
cum_med = np.cumsum(ev_median)
n_comp = len(ev_median)
x_range = np.arange(1, n_comp+1)

pca2 = PCA(n_components=2, random_state=42)
Xps2 = pca2.fit_transform(Xs)
km_labels = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xps2)
loadings = pca2.components_
feat_names = ALL_FEATURES

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('PCA Structure of the CERT r6.2 Weekly Feature Space',
             fontsize=12, fontweight='bold', y=1.01)

ax = axes[0]
ax.fill_between(x_range, np.cumsum(ev_q25), np.cumsum(ev_q75),
                 alpha=0.2, color='steelblue', label='IQR (16 weeks)')
ax.plot(x_range, cum_med, 'o-', color='steelblue', lw=2, ms=5, label='Median cumulative EV')
ax.axhline(0.90, ls='--', color='gray', lw=1, label='90% threshold')
ax.axhline(0.70, ls=':', color='gray', lw=1, label='70% threshold')
ax.axvline(2, ls='--', color='tomato', lw=1.5, label='n=2 (used)')
ax.scatter([2], [cum_med[1]], s=80, zorder=5, color='tomato')
ax.annotate(f'{cum_med[1]:.1%}', xy=(2, cum_med[1]), xytext=(3, cum_med[1]-0.06),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato')
ax.set_xlabel('Number of components'); ax.set_ylabel('Cumulative explained variance')
ax.set_title('Scree: cumulative explained variance\n(median +/- IQR, 16 sampled weeks)')
ax.set_xlim(0.5, n_comp+0.5); ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc='lower right'); ax.grid(True, alpha=0.3)

ax = axes[1]
colors = ['#4e79a7','#f28e2b','#59a14f','#e15759']
for c in range(DEFAULT_K):
    mask = km_labels == c
    ax.scatter(Xps2[mask, 0], Xps2[mask, 1], s=10, alpha=0.5, color=colors[c], label=f'Peer group {c+1}')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} EV)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} EV)')
ax.set_title(f'PC1-PC2 projection with K-Means (k={DEFAULT_K})\nWeek 2010-W20 (n={len(df_s):,})')
ax.legend(fontsize=8, markerscale=2); ax.grid(True, alpha=0.3)

ax = axes[2]
im = ax.imshow(np.abs(loadings), aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.6)
ax.set_xticks(range(len(feat_names))); ax.set_xticklabels(feat_names, rotation=45, ha='right', fontsize=7)
ax.set_yticks([0, 1]); ax.set_yticklabels(['PC1', 'PC2'], fontsize=9)
ax.set_title('|PCA loadings| per feature')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='|loading|')

plt.tight_layout()
plt.savefig('../results/figures/AWIK_PCA_Structure.pdf', bbox_inches='tight', dpi=300)
plt.savefig('../results/figures/AWIK_PCA_Structure.png', bbox_inches='tight', dpi=300)
plt.show()

## Stability check: k optimal across sampled weeks

In [ ]:
sample_weeks = sampled_weeks(weeks_df)
print(f"Sample weeks: {len(sample_weeks)} of {len(weeks_df)} total")

k_range_stab = range(2, 9)
stability_rows = []
for _, row in sample_weeks.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 20:
        continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    Xpw = PCA(n_components=2, random_state=42).fit_transform(Xw)
    sil_w = {}
    for k in k_range_stab:
        lb = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Xpw)
        sil_w[k] = silhouette_score(Xpw, lb)
    global_argmax = max(sil_w, key=sil_w.get)
    k4_is_local_max = sil_w[4] > sil_w.get(3,-1) and sil_w[4] > sil_w.get(5,-1)
    stability_rows.append({
        'Year_Week': f'{yr}-W{wk:02d}', 'n_users': len(df_w),
        **{f'sil_k{k}': round(v,4) for k,v in sil_w.items()},
        'global_argmax_k': global_argmax, 'k4_local_max': k4_is_local_max,
    })

df_stability = pd.DataFrame(stability_rows)
print(f"k=4 is local max (vs k=3 & k=5) in {df_stability['k4_local_max'].sum()}/"
      f"{len(df_stability)} weeks ({df_stability['k4_local_max'].mean()*100:.1f}%)")
print(f"Global argmax k distribution: {df_stability['global_argmax_k'].value_counts().sort_index().to_dict()}")

fig, ax = plt.subplots(figsize=(10,6))
for _, r in df_stability.iterrows():
    vals = [r[f'sil_k{k}'] for k in k_range_stab]
    ax.plot(list(k_range_stab), vals, alpha=0.4, marker='o', markersize=3)
ax.axvline(4, color='red', ls='--', label='k=4')
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title(f'k Stability Check: Silhouette per k, {len(df_stability)} sampled weeks')
ax.legend(); plt.tight_layout()
plt.savefig('../results/figures/AWIK_kStability_MultiWeek.png', dpi=150, bbox_inches='tight')
plt.show()
df_stability.to_csv('../results/tables/AWIK_kStability_MultiWeek.csv', index=False)

## DBSCAN parameters (minPts, eps) -- label-free

In [ ]:
D_SPACE = 2
MINPTS_RULE = 2 * D_SPACE
print(f"DBSCAN operates in {D_SPACE}D PCA space. Heuristic minPts=2*dim={MINPTS_RULE} "
      f"(used: DEFAULT_MINPTS={DEFAULT_MINPTS}, matches: {MINPTS_RULE == DEFAULT_MINPTS})")

# eps: within-cluster k-distance knee, across sampled weeks
eps_rows = []
for _, row in sampled_weeks(weeks_df).iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year'] == yr) & (df_master['Week'] == wk)]
    if len(df_w) < 50:
        continue
    Xpw = PCA(n_components=2, random_state=42).fit_transform(
        StandardScaler().fit_transform(df_w[ALL_FEATURES]))
    labels = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)
    kdist_pool = []
    for c in np.unique(labels):
        pts = Xpw[labels == c]
        if len(pts) <= DEFAULT_MINPTS:
            continue
        nn = NearestNeighbors(n_neighbors=DEFAULT_MINPTS).fit(pts)
        d, _ = nn.kneighbors(pts)
        kdist_pool.append(d[:, -1])
    if not kdist_pool:
        continue
    kdist = np.sort(np.concatenate(kdist_pool))
    _, eps_knee = knee_of_curve(kdist)
    eps_rows.append({'Year_Week': f'{yr}-W{wk:02d}', 'n_users': len(df_w), 'eps_knee': round(eps_knee, 4)})

df_eps = pd.DataFrame(eps_rows)
eps_med = df_eps['eps_knee'].median()
eps_lo, eps_hi = df_eps['eps_knee'].quantile(0.25), df_eps['eps_knee'].quantile(0.75)
print(f"eps_knee median={eps_med:.4f}, IQR=[{eps_lo:.4f}, {eps_hi:.4f}], "
      f"DEFAULT_EPS={DEFAULT_EPS} in IQR: {bool(eps_lo <= DEFAULT_EPS <= eps_hi)}")

# Sensitivity: noise fraction & cluster count vs eps, one representative week
mid = len(sampled_weeks(weeks_df)) // 2
sw2 = sampled_weeks(weeks_df)
yr_r, wk_r = int(sw2.iloc[mid]['Year']), int(sw2.iloc[mid]['Week'])
df_r = df_master[(df_master['Year'] == yr_r) & (df_master['Week'] == wk_r)]
Xpr = PCA(n_components=2, random_state=42).fit_transform(StandardScaler().fit_transform(df_r[ALL_FEATURES]))
lab_r = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpr)

print(f"\nSensitivity (week {yr_r}-W{wk_r:02d}): eps | noise% | n_clusters")
for eps in [0.20, 0.25, 0.30, 0.35, 0.39, 0.40, 0.45, 0.50]:
    noise_tot, ncl_tot = 0, 0
    for c in np.unique(lab_r):
        pts = Xpr[lab_r == c]
        if len(pts) <= DEFAULT_MINPTS:
            noise_tot += len(pts)
            continue
        dl = DBSCAN(eps=eps, min_samples=DEFAULT_MINPTS).fit_predict(pts)
        noise_tot += int(np.sum(dl == -1))
        ncl_tot += len(set(dl)) - (1 if -1 in dl else 0)
    marker = " <- used" if abs(eps - DEFAULT_EPS) < 1e-9 else ""
    print(f"  {eps:>6.2f} | {100*noise_tot/len(Xpr):>6.1f}% | {ncl_tot:>10d}{marker}")

kp = []
for c in np.unique(lab_r):
    pts = Xpr[lab_r == c]
    if len(pts) <= DEFAULT_MINPTS:
        continue
    nn = NearestNeighbors(n_neighbors=DEFAULT_MINPTS).fit(pts)
    d, _ = nn.kneighbors(pts)
    kp.append(d[:, -1])
kd = np.sort(np.concatenate(kp))
_, kv = knee_of_curve(kd)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(kd)), kd, lw=2)
ax.axhline(DEFAULT_EPS, color='red', ls='--', label=f'eps used = {DEFAULT_EPS}')
if kv is not None:
    ax.axhline(kv, color='green', ls=':', label=f'knee detected = {kv:.3f}')
ax.set_xlabel('Point (sorted ascending)'); ax.set_ylabel(f'Distance to {DEFAULT_MINPTS-1}-th neighbor')
ax.set_title(f'k-distance within-cluster - {yr_r}-W{wk_r:02d}'); ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/AWIK_epsKdistance.png', dpi=150, bbox_inches='tight')
plt.show()
df_eps.to_csv('../results/tables/AWIK_epsKnee_MultiWeek.csv', index=False)

## Cluster interpretation vs. Role/Department (not used to select k -- validation only)

In [ ]:
def purity(labels_pred, labels_true):
    df_p = pd.DataFrame({'c': labels_pred, 't': labels_true})
    return df_p.groupby('c')['t'].agg(lambda s: s.value_counts().iloc[0]).sum() / len(df_p)

print(f"Unique roles: {df_master['Role'].astype(str).nunique()}  Unique depts: {df_master['Department'].astype(str).nunique()}")

rows = []
for _, row in sampled_weeks(weeks_df).iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year'] == yr) & (df_master['Week'] == wk)]
    if len(df_w) < 50:
        continue
    Xpw = PCA(n_components=2, random_state=42).fit_transform(StandardScaler().fit_transform(df_w[ALL_FEATURES]))
    lab = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)
    role = df_w['Role'].astype(str).values
    dept = df_w['Department'].astype(str).values
    rows.append({
        'Year': yr, 'Week': wk, 'n_user': len(df_w),
        'ARI_role': adjusted_rand_score(role, lab), 'NMI_role': normalized_mutual_info_score(role, lab),
        'purity_role': purity(lab, role), 'base_role': pd.Series(role).value_counts(normalize=True).iloc[0],
        'ARI_dept': adjusted_rand_score(dept, lab), 'NMI_dept': normalized_mutual_info_score(dept, lab),
        'purity_dept': purity(lab, dept), 'base_dept': pd.Series(dept).value_counts(normalize=True).iloc[0],
    })
df_val = pd.DataFrame(rows)
for tgt in ['role', 'dept']:
    print(f"\n=== k={DEFAULT_K} vs {tgt.upper()} ===")
    for m in [f'ARI_{tgt}', f'NMI_{tgt}', f'purity_{tgt}', f'base_{tgt}']:
        q = df_val[m].quantile([.25, .5, .75])
        print(f"  {m:<12}: median={q[.5]:.3f}  IQR=[{q[.25]:.3f}, {q[.75]:.3f}]")

med_ari_r, med_ari_d = df_val['ARI_role'].median(), df_val['ARI_dept'].median()
print(f"\nGuide: ARI <0.05 none | 0.05-0.20 weak | 0.20-0.50 moderate | >0.50 strong")
print(f"Verdict: role={'correspondence' if med_ari_r >= 0.20 else 'weak/none'} "
      f"| dept={'correspondence' if med_ari_d >= 0.20 else 'weak/none'}")

## Multivariate normality diagnostic for the chi-square threshold

Tests the multivariate-normality assumption that used to underlie the theoretical
chi-square(df=2) threshold. Result: Shapiro-Wilk rejects normality for PC1 in most
cluster-weeks -- see `docs/methodology.md`. `mah_sq_pool` computed here is reused
by the next two cells.

In [ ]:
sample_weeks_norm = sampled_weeks(weeks_df)
mah_sq_pool = []
per_clusterweek_stat = []

for _, row in sample_weeks_norm.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50:
        continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    Xpw = PCA(n_components=2, random_state=42).fit_transform(Xw)
    lab = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)

    for cid in range(DEFAULT_K):
        pts = Xpw[lab == cid]
        if len(pts) < DEFAULT_MINPTS:
            continue
        db = DBSCAN(eps=DEFAULT_EPS, min_samples=DEFAULT_MINPTS).fit_predict(pts)
        clean = pts[db != -1]
        if len(clean) < 30:
            continue
        cov = np.cov(clean, rowvar=False) + np.eye(2) * 1e-6
        inv_cov = np.linalg.inv(cov)
        mean_c = np.mean(clean, axis=0)
        mah = np.array([mahalanobis(p, mean_c, inv_cov)**2 for p in clean])
        mah_sq_pool.append(mah)

        sw_pc1 = stats.shapiro(clean[:,0]) if len(clean) <= 5000 else (np.nan, np.nan)
        sw_pc2 = stats.shapiro(clean[:,1]) if len(clean) <= 5000 else (np.nan, np.nan)
        per_clusterweek_stat.append({
            'Year_Week': f'{yr}-W{wk:02d}', 'cluster': cid, 'n': len(clean),
            'shapiro_p_PC1': sw_pc1[1], 'shapiro_p_PC2': sw_pc2[1],
            'skew_PC1': round(stats.skew(clean[:,0]), 3), 'skew_PC2': round(stats.skew(clean[:,1]), 3),
            'kurt_excess_PC1': round(stats.kurtosis(clean[:,0]), 3),
            'kurt_excess_PC2': round(stats.kurtosis(clean[:,1]), 3),
        })

df_normality = pd.DataFrame(per_clusterweek_stat)
mah_sq_all = np.concatenate(mah_sq_pool)
print(f"Cluster-weeks tested: {len(df_normality)} | total points: {len(mah_sq_all):,}")
rej_pc1 = (df_normality['shapiro_p_PC1'] < 0.05).mean() * 100
rej_pc2 = (df_normality['shapiro_p_PC2'] < 0.05).mean() * 100
print(f"% cluster-weeks rejecting PC1 normality: {rej_pc1:.1f}%  | PC2: {rej_pc2:.1f}%")
print(f"Median skewness PC1/PC2: {df_normality['skew_PC1'].median():.2f} / {df_normality['skew_PC2'].median():.2f}")
print(f"Median excess kurtosis PC1/PC2: {df_normality['kurt_excess_PC1'].median():.2f} / {df_normality['kurt_excess_PC2'].median():.2f}")

try:
    import pingouin as pg
    hz_rows = []
    for _, row in sample_weeks_norm.iterrows():
        yr, wk = int(row['Year']), int(row['Week'])
        df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
        if len(df_w) < 50: continue
        Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
        Xpw = PCA(n_components=2, random_state=42).fit_transform(Xw)
        lab = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)
        for cid in range(DEFAULT_K):
            pts = Xpw[lab == cid]
            if len(pts) < 30: continue
            hz = pg.multivariate_normality(pts, alpha=.05)
            hz_rows.append({'Year_Week': f'{yr}-W{wk:02d}', 'cluster': cid, 'n': len(pts),
                             'HZ_stat': hz.hz, 'HZ_p': hz.pval, 'normal_at_0.05': hz.normal})
    df_hz = pd.DataFrame(hz_rows)
    print(f"Henze-Zirkler: % cluster-weeks NOT normal (p<0.05): {(~df_hz['normal_at_0.05']).mean()*100:.1f}%")
    df_hz.to_csv('../results/tables/AWIK_HenzeZirkler_PerClusterWeek.csv', index=False)
except ImportError:
    print("[pingouin not installed -- `pip install pingouin` for the formal Henze-Zirkler test]")

mah_sq_sorted = np.sort(mah_sq_all)
n = len(mah_sq_sorted)
theoretical_q = stats.chi2.ppf((np.arange(1, n+1) - 0.5) / n, df=2)

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(theoretical_q, mah_sq_sorted, s=4, alpha=0.3, color='steelblue')
lims = [0, max(theoretical_q.max(), mah_sq_sorted.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit to chi-square(df=2)')
ax.set_xlabel('Theoretical chi-square(df=2) quantile')
ax.set_ylabel('Observed Mahalanobis^2 quantile')
ax.set_title(f'Chi-Square Q-Q Plot\nn={n:,} points, {len(df_normality)} cluster-weeks')
ax.legend(); plt.tight_layout()
plt.savefig('../results/figures/AWIK_ChiSquare_QQPlot.png', dpi=150, bbox_inches='tight')
plt.show()
df_normality.to_csv('../results/tables/AWIK_Normality_PerClusterWeek.csv', index=False)

## Threshold: empirical percentile vs. Yeo-Johnson

Reuses `mah_sq_pool` from the previous cell. Yeo-Johnson does not fix normality
(kurtosis gets worse) -- rejected. See `docs/methodology.md`.

In [ ]:
sample_weeks_q = sampled_weeks(weeks_df)
rows_opt2, mah_sq_yj_pool, diag_yj_rows = [], [], []

for _, row in sample_weeks_q.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50:
        continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    Xpw = PCA(n_components=2, random_state=42).fit_transform(Xw)
    lab = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)

    Xw_yj = PowerTransformer(method='yeo-johnson', standardize=True).fit_transform(df_w[ALL_FEATURES])
    Xpw_yj = PCA(n_components=2, random_state=42).fit_transform(Xw_yj)
    lab_yj = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw_yj)

    for cid in range(DEFAULT_K):
        pts = Xpw[lab == cid]
        if len(pts) >= DEFAULT_MINPTS:
            db = DBSCAN(eps=DEFAULT_EPS, min_samples=DEFAULT_MINPTS).fit_predict(pts)
            clean = pts[db != -1]
            if len(clean) >= 30:
                cov = np.cov(clean, rowvar=False) + np.eye(2) * 1e-6
                inv_cov = np.linalg.inv(cov)
                mean_c = np.mean(clean, axis=0)
                mah_all = np.array([mahalanobis(p, mean_c, inv_cov)**2 for p in pts])
                emp_thr = np.percentile(mah_all, 99)
                rows_opt2.append({
                    'Year_Week': f'{yr}-W{wk:02d}', 'cluster': cid, 'n': len(pts),
                    'threshold_teoretis_9.21': round(9.2103, 3), 'threshold_empiris_p99': round(emp_thr, 3),
                    'rasio_empiris_vs_teoretis': round(emp_thr / 9.2103, 2),
                    'n_flag_teoretis': int((mah_all > 9.2103).sum()),
                    'n_flag_empiris_p99': int((mah_all > emp_thr).sum()),
                })
        pts_yj = Xpw_yj[lab_yj == cid]
        if len(pts_yj) >= DEFAULT_MINPTS:
            db_yj = DBSCAN(eps=DEFAULT_EPS, min_samples=DEFAULT_MINPTS).fit_predict(pts_yj)
            clean_yj = pts_yj[db_yj != -1]
            if len(clean_yj) >= 30:
                cov_yj = np.cov(clean_yj, rowvar=False) + np.eye(2) * 1e-6
                inv_cov_yj = np.linalg.inv(cov_yj)
                mean_yj = np.mean(clean_yj, axis=0)
                mah_yj = np.array([mahalanobis(p, mean_yj, inv_cov_yj)**2 for p in clean_yj])
                mah_sq_yj_pool.append(mah_yj)
                sw1 = stats.shapiro(clean_yj[:,0]) if len(clean_yj) <= 5000 else (np.nan, np.nan)
                sw2 = stats.shapiro(clean_yj[:,1]) if len(clean_yj) <= 5000 else (np.nan, np.nan)
                diag_yj_rows.append({
                    'Year_Week': f'{yr}-W{wk:02d}', 'cluster': cid, 'n': len(clean_yj),
                    'shapiro_p_PC1': sw1[1], 'shapiro_p_PC2': sw2[1],
                    'skew_PC1': round(stats.skew(clean_yj[:,0]), 3), 'skew_PC2': round(stats.skew(clean_yj[:,1]), 3),
                    'kurt_excess_PC1': round(stats.kurtosis(clean_yj[:,0]), 3),
                    'kurt_excess_PC2': round(stats.kurtosis(clean_yj[:,1]), 3),
                })

df_opt2 = pd.DataFrame(rows_opt2)
df_diag_yj = pd.DataFrame(diag_yj_rows)
print(f"Empirical p99 threshold: median={df_opt2['threshold_empiris_p99'].median():.2f}, "
      f"ratio vs theoretical: {df_opt2['rasio_empiris_vs_teoretis'].median():.2f}x")
print(f"Points flagged (theoretical 9.21): {df_opt2['n_flag_teoretis'].sum():,} | "
      f"(empirical p99): {df_opt2['n_flag_empiris_p99'].sum():,}")

rej_pc1_yj = (df_diag_yj['shapiro_p_PC1'] < 0.05).mean() * 100
rej_pc2_yj = (df_diag_yj['shapiro_p_PC2'] < 0.05).mean() * 100
print(f"\nAfter Yeo-Johnson: % rejecting PC1 normality: {rej_pc1_yj:.1f}% | PC2: {rej_pc2_yj:.1f}%")
print(f"Median excess kurtosis PC1/PC2: {df_diag_yj['kurt_excess_PC1'].median():.2f} / "
      f"{df_diag_yj['kurt_excess_PC2'].median():.2f}  (worse than before Yeo-Johnson -- rejected)")

mah_orig_clean_pool = []
for _, row in sample_weeks_q.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50: continue
    Xw = StandardScaler().fit_transform(df_w[ALL_FEATURES])
    Xpw = PCA(n_components=2, random_state=42).fit_transform(Xw)
    lab = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10).fit_predict(Xpw)
    for cid in range(DEFAULT_K):
        pts = Xpw[lab == cid]
        if len(pts) < DEFAULT_MINPTS: continue
        db = DBSCAN(eps=DEFAULT_EPS, min_samples=DEFAULT_MINPTS).fit_predict(pts)
        clean = pts[db != -1]
        if len(clean) < 30: continue
        cov = np.cov(clean, rowvar=False) + np.eye(2) * 1e-6
        inv_cov = np.linalg.inv(cov)
        mean_c = np.mean(clean, axis=0)
        mah_orig_clean_pool.append(np.array([mahalanobis(p, mean_c, inv_cov)**2 for p in clean]))

mah_orig_all = np.concatenate(mah_orig_clean_pool) if mah_orig_clean_pool else np.array([])
mah_yj_all = np.concatenate(mah_sq_yj_pool) if mah_sq_yj_pool else np.array([])

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, mah, title in [(axes[0], mah_orig_all, 'Before (StandardScaler only, clean points)'),
                         (axes[1], mah_yj_all, 'After Yeo-Johnson (clean points)')]:
    mah_sorted = np.sort(mah)
    n = len(mah_sorted)
    tq = stats.chi2.ppf((np.arange(1, n+1) - 0.5) / n, df=2)
    ax.scatter(tq, mah_sorted, s=4, alpha=0.3, color='steelblue')
    lims = [0, max(tq.max(), mah_sorted.max())]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Theoretical chi-square(df=2) quantile'); ax.set_ylabel('Observed Mahalanobis^2 quantile')
    ax.set_title(f'{title}\nn={n:,}')
plt.suptitle('Q-Q Plot: Yeo-Johnson impact on chi-square(df=2) fit (apple-to-apple, clean points only)')
plt.tight_layout()
plt.savefig('../results/figures/AWIK_QQPlot_YeoJohnson_Compare.png', dpi=150, bbox_inches='tight')
plt.show()

df_opt2.to_csv('../results/tables/AWIK_Opsi2_ThresholdEmpiris.csv', index=False)
df_diag_yj.to_csv('../results/tables/AWIK_Opsi3_NormalitasYeoJohnson.csv', index=False)
print(f"\n=> DEFAULT_MAH_THRESH cross-check: median(mah_orig_clean_pool) = "
      f"{np.median(mah_orig_all):.2f} (hardcoded in config.py: {DEFAULT_MAH_THRESH}; "
      f"update config.py if this diverges substantially on your run)")

## eps Stage-1 & Stage-2 for DBSCAN->DBSCAN (context for notebook 06)

Needed to test the DBSCAN->DBSCAN architecture candidate fairly (label-free), rather
than with arbitrary parameters. Result used in notebook 06, which ultimately rejects
this architecture.

In [ ]:
sample_weeks_dbdb = sampled_weeks(weeks_df)

eps_stage1_rows = []
for _, row in sample_weeks_dbdb.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50:
        continue
    Xpw = PCA(n_components=2, random_state=42).fit_transform(StandardScaler().fit_transform(df_w[ALL_FEATURES]))
    nn = NearestNeighbors(n_neighbors=DEFAULT_MINPTS).fit(Xpw)
    d, _ = nn.kneighbors(Xpw)
    kdist_global = np.sort(d[:, -1])
    _, knee_g = knee_of_curve(kdist_global)
    if knee_g is not None:
        eps_stage1_rows.append({'Year_Week': f'{yr}-W{wk:02d}', 'eps_knee_global': round(knee_g, 4)})

df_eps_stage1 = pd.DataFrame(eps_stage1_rows)
EPS_STAGE1 = df_eps_stage1['eps_knee_global'].median()
print(f"eps STAGE-1 (global, label-free): median={EPS_STAGE1:.4f}")

eps_stage2_rows = []
for _, row in sample_weeks_dbdb.iterrows():
    yr, wk = int(row['Year']), int(row['Week'])
    df_w = df_master[(df_master['Year']==yr) & (df_master['Week']==wk)]
    if len(df_w) < 50:
        continue
    Xpw = PCA(n_components=2, random_state=42).fit_transform(StandardScaler().fit_transform(df_w[ALL_FEATURES]))
    db1 = DBSCAN(eps=EPS_STAGE1, min_samples=DEFAULT_MINPTS).fit_predict(Xpw)
    kdist_pool = []
    for cid in set(db1):
        if cid == -1:
            continue
        pts = Xpw[db1 == cid]
        if len(pts) <= DEFAULT_MINPTS:
            continue
        nn = NearestNeighbors(n_neighbors=DEFAULT_MINPTS).fit(pts)
        d, _ = nn.kneighbors(pts)
        kdist_pool.append(d[:, -1])
    if not kdist_pool:
        continue
    kdist = np.sort(np.concatenate(kdist_pool))
    _, knee_s2 = knee_of_curve(kdist)
    if knee_s2 is not None:
        eps_stage2_rows.append({'Year_Week': f'{yr}-W{wk:02d}', 'eps_knee_stage2': round(knee_s2, 4)})

df_eps_stage2 = pd.DataFrame(eps_stage2_rows)
EPS_STAGE2 = df_eps_stage2['eps_knee_stage2'].median()
print(f"eps STAGE-2 (within-cluster, label-free): median={EPS_STAGE2:.4f}")
print(f"\nEPS_STAGE1={EPS_STAGE1:.4f}, EPS_STAGE2={EPS_STAGE2:.4f} -- used in notebook 06")

io.save_json({'EPS_STAGE1': EPS_STAGE1, 'EPS_STAGE2': EPS_STAGE2}, '02_eps_stage')
print("Checkpoint saved -> data/interim/02_eps_stage.json")